# Model Architecture - ChatKasir
- Nama: Achmad Rif'an
- Bagian: AI-1 (Model Architect)

## 1. Imports Library

In [10]:
import pandas as pd
import numpy as np
import tensorflow as tf

import os
from tokenizers import Tokenizer
from tokenizers.models import WordPiece
from tokenizers.trainers import WordPieceTrainer
from tokenizers.pre_tokenizers import Whitespace

# verifikasi versi
print(f"Pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")
print(f"TensorFlow: {tf.__version__}")

# check GPU yang tersedia
print(f"GPU tersedia: {len(tf.config.list_physical_devices('GPU')) > 0}")

Pandas: 3.0.2
NumPy: 2.4.4
TensorFlow: 2.21.0
GPU tersedia: False


## 2. Data Loading

In [19]:
# URL Dataset dari DS-1
url_food = "https://drive.google.com/uc?id=1xpoFjqAT9K0uwzSpVADm_EfKqG7dxVUI"
url_slang = "https://drive.google.com/uc?id=1G14C1qcqOp06Xs1HFiorE3Us_LLtaBs7"
url_sintetis = "https://drive.google.com/uc?id=17lFTivPH4BXEd6zoa-qjpUohNmh1ELdo"

# Fungsi membaca CSV dari Google Drive
def load_gdrive_csv(url):
    return pd.read_csv(url)

# Memuat ke dalam DataFrame
df_food = load_gdrive_csv(url_food)
df_slang = load_gdrive_csv(url_slang)
df_sintetis = load_gdrive_csv(url_sintetis)

print(f"Total data chat sintetis: {len(df_sintetis)} baris")
print(f"Total daftar menu: {len(df_food)} baris")

Total data chat sintetis: 100500 baris
Total daftar menu: 18558 baris


In [ ]:
# Tampilkan 5 baris pertama data food dan chat sintetis
display(df_food.head())
display(df_sintetis.head())

,name
0,abon
1,abon ayam
2,abon burger
3,abon cheese burger
4,abon goreng ayam


,input_text,product,quantity,price_satuan,pattern
0,kk mau pesen 10 daebak ken chicken wings [SEP]...,daebak ken chicken wings,10,3000,2
1,kk 3 nasi ayam betutu ya kak [SEP] noted kak n...,nasi ayam betutu,3,60000,2
2,bu mau pesen 6 indomie seafood [SEP] indomie s...,indomie seafood,6,37000,3
3,bg 9 chicken double dong [SEP] baik kak,chicken double,9,-1,1
4,bg bisa pesan 7 mie doer dong [SEP] oke mie do...,mie doer,7,42000,2


## 3. Tokenizer

In [28]:
# Gabungkan teks chat sintetis dan nama makanan
semua_teks = df_sintetis['input_text'].astype(str).tolist() + df_food['name'].astype(str).tolist()

# Inisialisasi Tokenizer WordPiece dengan token [UNK] untuk kata tak dikenal
tokenizer = Tokenizer(WordPiece(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()  # Pisahkan kata berdasarkan spasi

# Atur Trainer: vocab 5000 untuk domain UMKM
# Special tokens:
# - [PAD]: padding
# - [UNK]: kata tak dikenal
# - [SEP]: pemisah chat pembeli dan penjual
trainer = WordPieceTrainer(
    vocab_size=5000, 
    special_tokens=["[PAD]", "[UNK]", "[SEP]"]
)

# Latih Tokenizer dengan teks gabungan
tokenizer.train_from_iterator(semua_teks, trainer)

# Simpan tokenizer ke file JSON
os.makedirs("..\\assets\\tokenizers", exist_ok=True)
tokenizer.save("..\\assets\\tokenizers\\tokenizer.json")
print("Tokenizer disimpan di '..\\assets\\tokenizers\\tokenizer.json'")

# Ambil ukuran kosakata akhir
vocab_size = tokenizer.get_vocab_size()
print(f"Ukuran Vocab: {vocab_size}")

# Tes kemampuan tokenizer menangani typo
tes_kalimat = "kk mau pesen 10 daebak ken chicken wings [SEP] baik kak daebak ken chicken wings rp3rb satuan totalnya rp30rb"
hasil_tes = tokenizer.encode(tes_kalimat)

print(f"\n===HASIL TES TOKENIZER===")
print(f"Kalimat asli: {tes_kalimat}")
print(f"Dipecah menjadi tokens: {hasil_tes.tokens}")
print(f"Diubah ke ID angka: {hasil_tes.ids}")


Tokenizer disimpan di '..\assets\tokenizers\tokenizer.json'
Ukuran Vocab: 5000

===HASIL TES TOKENIZER===
Kalimat asli: kk mau pesen 10 daebak ken chicken wings [SEP] baik kak daebak ken chicken wings rp3rb satuan totalnya rp30rb
Dipecah menjadi tokens: ['kk', 'mau', 'pesen', '10', 'daebak', 'ken', 'chicken', 'wings', '[SEP]', 'baik', 'kak', 'daebak', 'ken', 'chicken', 'wings', 'rp3rb', 'satuan', 'totalnya', 'rp30rb']
Diubah ke ID angka: [308, 125, 175, 172, 1379, 1318, 144, 358, 2, 164, 84, 1379, 1318, 144, 358, 1690, 177, 105, 1309]


## 4. Menghitung Max Length

In [32]:
# Hitung panjang token dari setiap baris di dataset chat sintetis
panjang_semua_teks = [len(tokenizer.encode(teks).ids) for teks in df_sintetis['input_text'].astype(str)]

# Cari yang paling panjang
max_length = max(panjang_semua_teks)

print(f"Panjang kalimat maksimal di dataset (max_length): {max_length}")

# Membulatkan max_length ke 64
optimal_max_length = 64 if max_length < 64 else max_length
print(f"Max Length optimal yang akan digunakan: {optimal_max_length}")

Panjang kalimat maksimal di dataset (max_length): 31
Max Length optimal yang akan digunakan: 64
